# Day 07: Multi-Layer Network — Solving XOR

**Goal:** Stack 2 layers of neurons and finally crack XOR.

### Where we left off

Day 06: a single neuron solved AND and OR, but failed at XOR. The reason: XOR is not **linearly separable** — no single straight line can separate the 1s from the 0s.

### Today's solution

Stack neurons into layers. Two layers can create non-linear decision boundaries.

```
   Day 06: ONE neuron              Day 07: TWO layers
   ┌─────┐                         ┌─────┐
   │     │                         │  h1 │───┐
x ─▶│  N  │──▶ y               x ─▶│     │   │
   │     │                         └─────┘   ▼
   └─────┘                         ┌─────┐ ┌────┐
   Draws ONE line                  │  h2 │─▶│ out│──▶ y
   (can't do XOR)              x ─▶│     │ └────┘
                                   └─────┘
                                   Draws COMBINATIONS of lines
                                   (can solve XOR!)
```

Today we'll build this TWICE:
1. **From scratch** — manual backpropagation (to understand what happens)
2. **Using `nn.Module`** — the clean way (what we'll use forever after)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

# XOR data
X = torch.tensor([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])
y = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

print("XOR truth table:")
for i in range(4):
    print(f"  {X[i, 0].item():.0f} XOR {X[i, 1].item():.0f} = {y[i, 0].item():.0f}")

## 1. The Architecture

A simple 2-layer network:

```
INPUT (2)  →  HIDDEN (2 neurons, ReLU)  →  OUTPUT (1 neuron, sigmoid)

Parameters:
  W1: (2×2) matrix + b1: (2,) bias   →  6 params
  W2: (2×1) matrix + b2: (1,) bias   →  3 params
  Total: 9 parameters
```

Each layer is a matrix multiplication followed by an activation.

### The forward pass:

```python
z1 = X @ W1 + b1        # (4, 2) @ (2, 2) = (4, 2)
h  = relu(z1)           # (4, 2) — hidden layer activations
z2 = h @ W2 + b2        # (4, 2) @ (2, 1) = (4, 1)
y  = sigmoid(z2)        # (4, 1) — final predictions
```

Trace the shapes: 4 samples, 2 input features, 2 hidden neurons, 1 output.

## 2. Build It From Scratch — Manual Forward + Backward

Let's write the network WITHOUT `nn.Module`, so you can see every multiplication and every gradient.

### Manual backpropagation formulas:

We'll use sigmoid for both layers to simplify the math. The gradients work out nicely:

```
y = sigmoid(z2)
loss = BCE(y, target)

∂loss/∂z2 = y - target                 (beautiful simplification!)
∂loss/∂W2 = h.T @ ∂loss/∂z2             (chain rule: gradient × input)
∂loss/∂b2 = sum of ∂loss/∂z2 over samples

∂loss/∂h  = ∂loss/∂z2 @ W2.T            (push gradient back)
∂loss/∂z1 = ∂loss/∂h × sigmoid'(z1)     (derivative of sigmoid)
∂loss/∂W1 = X.T @ ∂loss/∂z1
∂loss/∂b1 = sum of ∂loss/∂z1 over samples
```

Don't worry about memorizing these — the point is to SEE that it's just chain rule applied mechanically. PyTorch will handle this automatically in the next section.

In [ ]:
# Build a 2-layer network FROM SCRATCH (no autograd!)
# Everything computed by hand — all for learning purposes

torch.manual_seed(0)

# Initialize weights randomly (small values to break symmetry)
W1 = torch.randn(2, 2) * 0.5   # (input_dim, hidden_dim)
b1 = torch.zeros(1, 2)          # (1, hidden_dim) — broadcasts
W2 = torch.randn(2, 1) * 0.5   # (hidden_dim, output_dim)
b2 = torch.zeros(1, 1)

def sigmoid(z):
    return 1 / (1 + torch.exp(-z))

lr = 1.0
losses_manual = []

for epoch in range(10000):
    # ---- FORWARD PASS ----
    z1 = X @ W1 + b1              # (4, 2)
    h  = sigmoid(z1)               # (4, 2)
    z2 = h @ W2 + b2               # (4, 1)
    y_pred = sigmoid(z2)           # (4, 1)
    
    # ---- LOSS: Binary Cross-Entropy ----
    eps = 1e-8  # avoid log(0)
    loss = -(y * torch.log(y_pred + eps) + (1 - y) * torch.log(1 - y_pred + eps)).mean()
    losses_manual.append(loss.item())
    
    # ---- BACKWARD PASS (all by hand!) ----
    # Using the shortcut: ∂loss/∂z2 = y_pred - y (derived from BCE+sigmoid)
    dz2 = (y_pred - y) / 4         # divide by N for mean loss
    
    # Output layer gradients
    dW2 = h.T @ dz2                # (2, 4) @ (4, 1) = (2, 1)
    db2 = dz2.sum(dim=0, keepdim=True)
    
    # Push gradient back to hidden layer
    dh = dz2 @ W2.T                # (4, 1) @ (1, 2) = (4, 2)
    dz1 = dh * h * (1 - h)         # sigmoid'(z1) = h * (1 - h)
    
    # Hidden layer gradients
    dW1 = X.T @ dz1                # (2, 4) @ (4, 2) = (2, 2)
    db1 = dz1.sum(dim=0, keepdim=True)
    
    # ---- UPDATE ----
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

# Final predictions
with torch.no_grad():
    z1 = X @ W1 + b1
    h  = sigmoid(z1)
    z2 = h @ W2 + b2
    pred = sigmoid(z2)

print("Manual 2-layer network (NO AUTOGRAD):")
print(f"\n{'Input':>10} | {'Target':>6} | {'Predicted':>9} | {'Correct?':>8}")
print("-" * 50)
for i in range(4):
    x1, x2 = X[i, 0].item(), X[i, 1].item()
    t = y[i, 0].item()
    p = pred[i, 0].item()
    correct = "✓" if round(p) == t else "✗"
    print(f"  ({x1:.0f}, {x2:.0f})   | {t:>6.0f} | {p:>9.3f} | {correct:>8}")

print(f"\nFinal loss: {losses_manual[-1]:.6f}")
print("✓ XOR solved — by a 2-layer network with manual backprop!")

### XOR is SOLVED!

A single neuron was stuck at loss 0.69 (random guessing). This 2-layer network drops loss close to 0 — it learned XOR.

**What changed?** Just one extra layer. The hidden layer creates intermediate features, and the output layer combines them to produce the non-linear XOR pattern.

We wrote every gradient by hand. PyTorch can do this automatically:

## 3. Now With `nn.Module` — The Clean Way

Same network, much less code. This is what you'll write in real projects:

In [ ]:
# Same network using PyTorch's building blocks

class XORNet(nn.Module):
    def __init__(self, hidden_dim=2):
        super().__init__()
        self.hidden = nn.Linear(2, hidden_dim)   # Layer 1: 2 → hidden_dim
        self.output = nn.Linear(hidden_dim, 1)   # Layer 2: hidden_dim → 1
    
    def forward(self, x):
        h = torch.sigmoid(self.hidden(x))        # hidden activations
        y = torch.sigmoid(self.output(h))        # output probability
        return y

torch.manual_seed(0)
model = XORNet(hidden_dim=2)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model architecture: 2 → 2 → 1")
print(f"Total parameters: {total_params}\n")

# Train
optimizer = torch.optim.SGD(model.parameters(), lr=1.0)
losses_auto = []

for epoch in range(10000):
    y_pred = model(X)
    loss = nn.functional.binary_cross_entropy(y_pred, y)
    losses_auto.append(loss.item())
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Check predictions
with torch.no_grad():
    predictions = model(X)

print("XORNet (using nn.Module + autograd):")
print(f"\n{'Input':>10} | {'Target':>6} | {'Predicted':>9} | {'Correct?':>8}")
print("-" * 50)
for i in range(4):
    x1, x2 = X[i, 0].item(), X[i, 1].item()
    t = y[i, 0].item()
    p = predictions[i, 0].item()
    correct = "✓" if round(p) == t else "✗"
    print(f"  ({x1:.0f}, {x2:.0f})   | {t:>6.0f} | {p:>9.3f} | {correct:>8}")

print(f"\nFinal loss: {losses_auto[-1]:.6f}")
print("\nSame result — just way less code!")

In [ ]:
# Compare loss curves: manual vs autograd (should be nearly identical)

plt.figure(figsize=(10, 5))
plt.plot(losses_manual, 'b-', label='Manual backprop (from scratch)', linewidth=2, alpha=0.7)
plt.plot(losses_auto, 'r--', label='PyTorch autograd', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss (BCE)')
plt.title('XOR: Manual vs PyTorch — both work, autograd is easier')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Both curves drop to near-zero loss → XOR is learnable with 2 layers.")
print("\nManual approach: we wrote ~20 lines of gradient math.")
print("Autograd approach: we wrote ~5 lines and PyTorch figured it all out.")
print("\nFrom now on we use autograd. The manual version was for understanding.")

## 4. The Decision Boundary — Now Non-Linear!

In Day 06 the single neuron drew a straight line. Now let's see what a 2-layer network produces:

In [ ]:
# Plot the decision boundary

def plot_decision_boundary(model, X, y, title='Decision Boundary'):
    # Create a fine grid
    x_grid = torch.linspace(-0.3, 1.3, 200)
    y_grid = torch.linspace(-0.3, 1.3, 200)
    xx, yy = torch.meshgrid(x_grid, y_grid, indexing='xy')
    grid_points = torch.stack([xx.flatten(), yy.flatten()], dim=1)
    
    with torch.no_grad():
        z_grid = model(grid_points).reshape(xx.shape)
    
    fig, ax = plt.subplots(figsize=(8, 7))
    
    # Contour of probability
    contour = ax.contourf(xx.numpy(), yy.numpy(), z_grid.numpy(), 
                           levels=20, cmap='RdBu_r', alpha=0.6)
    plt.colorbar(contour, ax=ax, label='P(y=1)')
    
    # Decision boundary at 0.5
    ax.contour(xx.numpy(), yy.numpy(), z_grid.numpy(),
               levels=[0.5], colors='black', linewidths=2)
    
    # Data points
    for i in range(4):
        color = 'red' if y[i] == 1 else 'blue'
        marker = 'o' if y[i] == 1 else 's'
        ax.scatter(X[i, 0].item(), X[i, 1].item(), s=500, c=color, marker=marker,
                   edgecolor='white', linewidth=3, zorder=5)
        ax.annotate(f'({int(X[i,0].item())},{int(X[i,1].item())})',
                    (X[i,0].item(), X[i,1].item()),
                    textcoords="offset points", xytext=(15, 15), fontsize=11, 
                    color='white', weight='bold')
    
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_decision_boundary(model, X, y, title='2-layer network: non-linear XOR boundary')

print("Look at the BLACK decision boundary — it's CURVED, not straight!")
print("The network carved out two red regions (where XOR = 1).")
print("This is what hidden layers enable: non-linear decision boundaries.")

## 5. What Did Each Hidden Neuron Learn?

Let's peek inside and see what each of the 2 hidden neurons is doing:

In [ ]:
# Visualize what each hidden neuron outputs

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Compute hidden neuron outputs on a grid
x_grid = torch.linspace(-0.3, 1.3, 200)
y_grid = torch.linspace(-0.3, 1.3, 200)
xx, yy = torch.meshgrid(x_grid, y_grid, indexing='xy')
grid_points = torch.stack([xx.flatten(), yy.flatten()], dim=1)

with torch.no_grad():
    hidden_outputs = torch.sigmoid(model.hidden(grid_points))   # shape (40000, 2)
    final_output = model(grid_points).reshape(xx.shape)

# Plot each hidden neuron's output
for i in range(2):
    z = hidden_outputs[:, i].reshape(xx.shape)
    ax = axes[i]
    contour = ax.contourf(xx.numpy(), yy.numpy(), z.numpy(),
                           levels=20, cmap='Purples', alpha=0.8)
    plt.colorbar(contour, ax=ax)
    
    for j in range(4):
        color = 'red' if y[j] == 1 else 'blue'
        marker = 'o' if y[j] == 1 else 's'
        ax.scatter(X[j, 0].item(), X[j, 1].item(), s=300, c=color, marker=marker,
                   edgecolor='white', linewidth=2, zorder=5)
    
    w1_neuron = model.hidden.weight.data[i]
    b1_neuron = model.hidden.bias.data[i]
    ax.set_title(f'Hidden Neuron {i+1} output\nw=[{w1_neuron[0]:.2f}, {w1_neuron[1]:.2f}], b={b1_neuron.item():.2f}')
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    ax.grid(True, alpha=0.3)

# Final combined output
ax = axes[2]
contour = ax.contourf(xx.numpy(), yy.numpy(), final_output.numpy(),
                       levels=20, cmap='RdBu_r', alpha=0.8)
plt.colorbar(contour, ax=ax)
ax.contour(xx.numpy(), yy.numpy(), final_output.numpy(),
           levels=[0.5], colors='black', linewidths=2)

for j in range(4):
    color = 'red' if y[j] == 1 else 'blue'
    marker = 'o' if y[j] == 1 else 's'
    ax.scatter(X[j, 0].item(), X[j, 1].item(), s=300, c=color, marker=marker,
               edgecolor='white', linewidth=2, zorder=5)

ax.set_title('Final output\n(combines the 2 hidden neurons)')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Each hidden neuron draws a different LINE (since each is linear + sigmoid).")
print("The output neuron COMBINES them to carve out non-linear regions.")
print("\nThis is the magic: 2 lines can combine to make complex boundaries.")

## 6. Universal Approximation — Fit an Arbitrary Curve

The Universal Approximation Theorem says: a network with ONE hidden layer can approximate ANY continuous function. Let's test this by fitting a complex curve:

In [ ]:
# Fit an arbitrary curve: y = sin(x) + 0.3*cos(3x)
# A single neuron could never do this. A 2-layer net with enough hidden neurons can!

torch.manual_seed(42)

# Generate data
x_data = torch.linspace(-3, 3, 200).unsqueeze(1)
y_data = torch.sin(x_data) + 0.3 * torch.cos(3 * x_data)

# A simple 2-layer network
class CurveFitter(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(1, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)
    
    def forward(self, x):
        h = torch.tanh(self.fc1(x))    # tanh works well for regression
        return self.fc2(h)

# Try three different sizes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, hidden_dim in zip(axes, [3, 10, 50]):
    torch.manual_seed(42)
    model_c = CurveFitter(hidden_dim=hidden_dim)
    optimizer = torch.optim.Adam(model_c.parameters(), lr=0.01)
    
    for epoch in range(3000):
        y_pred = model_c(x_data)
        loss = ((y_pred - y_data) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    with torch.no_grad():
        y_learned = model_c(x_data)
    
    ax.scatter(x_data.numpy(), y_data.numpy(), alpha=0.3, s=15, label='True data', color='steelblue')
    ax.plot(x_data.numpy(), y_learned.numpy(), 'r-', linewidth=2, label='Learned')
    ax.set_title(f'{hidden_dim} hidden neurons\nfinal loss: {loss.item():.4f}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("With 3 neurons:  model is too small, can only approximate roughly")
print("With 10 neurons: good fit for the main pattern")
print("With 50 neurons: very close match to the true function")
print("\nUNIVERSAL APPROXIMATION: more neurons → can approximate ANY function.")

## 7. Why Random Initialization Matters

If all weights start at zero, all hidden neurons compute identically and stay that way. Let's prove it:

In [ ]:
# Train XOR with ZERO init vs RANDOM init

def train_xor_with_init(init_type='random', hidden_dim=4, epochs=5000):
    torch.manual_seed(0)
    model = XORNet(hidden_dim=hidden_dim)
    
    if init_type == 'zero':
        # Zero-out all weights (terrible idea!)
        for p in model.parameters():
            torch.nn.init.zeros_(p)
    # else: keep PyTorch's default random init
    
    optimizer = torch.optim.SGD(model.parameters(), lr=1.0)
    losses = []
    for _ in range(epochs):
        y_pred = model(X)
        loss = nn.functional.binary_cross_entropy(y_pred, y)
        losses.append(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return losses

losses_zero = train_xor_with_init('zero')
losses_random = train_xor_with_init('random')

plt.figure(figsize=(10, 5))
plt.plot(losses_zero, 'r-', label='Zero initialization (BROKEN)', linewidth=2)
plt.plot(losses_random, 'b-', label='Random initialization (works)', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Initialization matters: zero init breaks learning')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Zero init:   loss stays at 0.69 — network can't break symmetry → fails")
print("Random init: loss drops to near 0 — network learns XOR")
print("\nLesson: always initialize weights randomly (PyTorch does this by default).")

---

## Exercises

1. **Change hidden size:** Set `hidden_dim=1`. Can XOR be learned with just 1 hidden neuron? What about `hidden_dim=10`?

2. **Try different activations:** Use `nn.ReLU()` in the hidden layer instead of sigmoid. Does XOR still train? Which is faster?

3. **Add a third layer:** Modify `XORNet` to have 3 linear layers (input → hidden1 → hidden2 → output). Does training still work?

4. **Watch training progress:** Plot the decision boundary every 1000 epochs. You'll see it gradually carve out the XOR pattern.

5. **Break it:** Use a learning rate of 100. Does it explode? Does it ever converge?

---

## Key Takeaways

### What a 2-layer network can do:
- Solve XOR (single neuron can't)
- Fit arbitrary curves (Universal Approximation Theorem)
- Create non-linear decision boundaries

### Network architecture:
```
Input → Linear(W1, b1) → Activation → Linear(W2, b2) → Activation → Output
        \_____________________________________________________________/
                             forward pass
```

### The training loop (forever and always):
```python
y_pred = model(X)                # forward
loss = loss_fn(y_pred, y)        # measure
optimizer.zero_grad()            # clear old gradients
loss.backward()                   # compute new gradients (autograd!)
optimizer.step()                  # update all parameters
```

### Key insights:
- **Each layer transforms its input** → extracts more abstract features
- **Backprop = chain rule** applied backwards through layers (PyTorch does this)
- **Random init is essential** → breaks symmetry between neurons
- **More layers/neurons = more power** (up to a point, then diminishing returns)

### Historical context:
- 1958: Perceptron (single neuron) invented
- 1969: Minsky's book proves single neurons can't learn XOR → AI winter
- 1986: Backprop rediscovered for multi-layer nets → AI comeback
- 2012: Deep learning revolution (GPUs enable huge networks)
- 2024: GPT-4, Claude, etc. — essentially MASSIVE versions of today's architecture

**Tomorrow:** We'll build a proper training loop with validation, checkpointing, and learning rate tricks (Day 08).